# Monkey Patching in Python

---

## 1. Simple Intuition First

You're using a third-party payment SDK in production. It has a bug. Vendor won't fix it for 2 weeks. You can't change their source code.

**What do you do?** You quietly swap out their broken function with your fixed one — at runtime — without touching their package files.

That's monkey patching.

> **One-liner:** Replace or extend code behavior at runtime by reassigning names in Python's module/class namespace — no source file changes needed.

Python makes this possible because **everything is just a name pointing to an object in a dictionary**. You can always change what a name points to.

```python
import requests

# requests.get currently points to the real HTTP function
requests.get = my_fake_function
# now ALL code calling requests.get hits your function
```

---

## 2. Why This Topic Exists

| Problem | How Monkey Patching Solves It |
|---|---|
| Third-party library bug, can't wait for fix | Swap the broken function at runtime |
| Testing without real DB/APIs | Replace real calls with mocks |
| High concurrency without rewriting code | gevent replaces blocking I/O stdlib-wide |
| Observability without touching app code | APM agents (Datadog) inject tracing into your DB/HTTP calls |

**In production, you see it in three places:**
1. `unittest.mock.patch` — testing (most common)
2. `gevent.monkey.patch_all()` — high concurrency Gunicorn workers
3. APM auto-instrumentation — Datadog, New Relic silently patch your `psycopg2`, `redis` calls

---

## 3. Core Concepts

### 3.1 How It Works — The Dict Trick

Python modules and classes store their attributes in `__dict__`. Patching = writing to that dict.

```python
import os
os.__dict__['getenv'] = my_fake_getenv
# same as:
os.getenv = my_fake_getenv
```

### 3.2 `unittest.mock.patch` — The Interview-Critical One

The right way to patch in tests. **Auto-restores** the original after the block exits.

```python
from unittest.mock import patch, MagicMock

# As decorator
@patch('payment_service.stripe_client.charge')
def test_payment(mock_charge):
    mock_charge.return_value = {"status": "success", "id": "txn_123"}
    result = payment_service.process(100)
    assert result["id"] == "txn_123"
    mock_charge.assert_called_once_with(amount=100)

# As context manager
with patch('app.cache.redis_client.get') as mock_get:
    mock_get.return_value = None  # simulate cache miss
    result = fetch_user_profile("U001")
```

### 3.3 The Golden Rule — Patch Where It's USED

This is the single most tested concept in interviews.

```python
# payment_service.py
from requests import post   # 'post' is now a LOCAL name in this module

# WRONG — changes requests module, but payment_service.post still points to original
@patch('requests.post')

# CORRECT — changes the name inside payment_service's namespace
@patch('payment_service.post')
```

**Rule:** `from x import func` → patch `your_module.func`
**Rule:** `import x; x.func()` → patch `x.func`

### 3.4 `patch` Variants

```python
# 1. patch() — string path, most common
@patch('app.services.stripe.Charge.create')

# 2. patch.object() — when you have direct reference to class
@patch.object(StripeClient, 'charge')

# 3. patch.dict() — for dicts like os.environ, feature flags
@patch.dict(os.environ, {'STRIPE_KEY': 'sk_test_fake', 'ENV': 'test'})
```

### 3.5 `MagicMock` — Always Use `spec=`

```python
# Without spec — dangerous, phantom attributes silently return mocks
mock_session = MagicMock()
mock_session.nonexistent_method()  # returns Mock instead of failing!

# With spec — safe, only real methods allowed
mock_session = MagicMock(spec=Session)
mock_session.nonexistent_method()  # AttributeError — test correctly fails
```

### 3.6 `gevent.monkey.patch_all()` — Must Know for Concurrency Questions

```python
# wsgi.py — must be FIRST lines before any other import
import gevent.monkey
gevent.monkey.patch_all()

from app import create_app
app = create_app()
```

What it patches: `socket`, `ssl`, `threading`, `time.sleep` → replaced with non-blocking greenlet versions.
Result: one worker thread handles thousands of concurrent I/O requests instead of one.

---

## 4. Production-Level Example

### Testing a Payment Service (Most Common Interview Scenario)

```python
# payment_service.py
import razorpay

client = razorpay.Client(auth=(KEY_ID, SECRET))

def charge_customer(amount_paise: int, token: str) -> dict:
    order = client.order.create({"amount": amount_paise, "currency": "INR"})
    return {"order_id": order["id"], "status": order["status"]}
```

```python
# test_payment_service.py
from unittest.mock import patch, MagicMock
import pytest

class TestChargeCustomer:

    @patch('payment_service.client.order.create')
    def test_successful_charge(self, mock_create):
        mock_create.return_value = {"id": "order_xyz", "status": "created"}

        result = charge_customer(50000, "tok_test")

        assert result["order_id"] == "order_xyz"
        mock_create.assert_called_once_with({"amount": 50000, "currency": "INR"})

    @patch('payment_service.client.order.create')
    def test_gateway_timeout(self, mock_create):
        mock_create.side_effect = razorpay.errors.ServerError("timeout")

        with pytest.raises(razorpay.errors.ServerError):
            charge_customer(50000, "tok_test")
```

### APM Auto-Instrumentation (Where It Lives Silently in Prod)

When you run `ddtrace-run python app.py`, Datadog patches your DB drivers before your app starts:

```python
# What ddtrace does internally — you never write this, but know it exists
_original_execute = psycopg2.extensions.cursor.execute

def traced_execute(self, query, vars=None):
    with tracer.start_span('postgresql.query') as span:
        span.set_tag('db.statement', query[:200])
        return _original_execute(self, query, vars)  # call real function

psycopg2.extensions.cursor.execute = traced_execute
# Every DB query in your entire app now gets a trace span automatically
```

---

## 5. Internal Working

**When you call `@patch('payment_service.stripe.charge')`:**

```
1. Python imports 'payment_service' module
2. Navigates to 'stripe.charge' attribute on that module
3. Saves original: _orig = payment_service.stripe.charge
4. Replaces:        payment_service.stripe.charge = MagicMock()
5. Runs your test
6. Restores:        payment_service.stripe.charge = _orig
```

**Why attribute lookup makes this work:**

```
instance.method()
  → check instance.__dict__   (not there)
  → check Class.__dict__      (finds patched version)
  → executes patched method
```

**Why gevent must be patched first:**

```python
import requests          # requests grabs original socket → holds direct reference
import gevent.monkey
gevent.monkey.patch_all()  # TOO LATE — requests already has old socket

# Correct:
import gevent.monkey
gevent.monkey.patch_all()  # updates sys.modules['socket']
import requests           # now gets patched socket ✓
```

---

## 6. Most Common Interview Questions

---

### Q1: What is monkey patching? Give a real example.

**Answer:**
> Replacing code behavior at runtime by reassigning names in Python's module/class namespace without touching source files. Three real uses: `mock.patch` for testing, `gevent.patch_all()` for high-concurrency workers, and APM agents like Datadog that inject distributed tracing into your DB calls automatically.

**Follow-up:** *"What are the risks?"*
> Invisible behavior change (hard to debug), version fragility (library internals change → patch breaks silently), test pollution if patch not properly scoped.

**Mistake:** Only mentioning testing/mocks. Not knowing gevent or APM use cases.

---

### Q2: Why do you patch where the name is used, not defined?

**Answer:**
> Because `from requests import post` creates a **local copy of the name** in your module. Patching `requests.post` changes the original module but your local binding still points to the old function. You must patch `your_module.post` to change what your code actually calls.

```python
# payment.py does: from requests import post
@patch('payment.post')       # ✓ correct
@patch('requests.post')      # ✗ wrong — payment.post unaffected
```

**Mistake:** Almost every candidate gets this wrong once. Show you know the *why*.

---

### Q3: `patch` vs `patch.object` vs `patch.dict`?

**Answer:**
- `patch('module.attr')` — string path, patches attribute on a module
- `patch.object(Class, 'method')` — direct class reference, no string fragility
- `patch.dict(os.environ, {...})` — patches dictionary entries, auto-restores

**When to use each:** `patch` for most cases. `patch.object` when you have the class in hand. `patch.dict` always for env vars.

---

### Q4: Why use `spec=` in MagicMock?

**Answer:**
> Without `spec`, `mock.any_typo()` silently returns a Mock instead of failing. Your test passes but is testing nothing real. `spec=RealClass` constrains the mock to only have attributes the real class has — typos and API mismatches cause `AttributeError` immediately.

```python
mock = MagicMock(spec=RedisClient)
mock.gte("key")   # AttributeError — caught in test, not production
```

---

### Q5: How does gevent's monkey patching enable high concurrency?

**Answer:**
> Standard Python `socket` calls are blocking — one request blocks the thread. `gevent.patch_all()` replaces `socket`, `ssl`, `threading`, `time.sleep` with non-blocking greenlet-based versions. When one greenlet waits for I/O, others run. Result: one Gunicorn worker handles 1000 concurrent connections instead of 1. No application code changes needed — even third-party libraries get the benefit because stdlib is patched globally.

---

## 7. Senior-Level Understanding

### When to Use vs Avoid

| Use it | Avoid it |
|---|---|
| Tests — mocking external services | New code — use dependency injection instead |
| gevent concurrency | Long-term "fixes" for library bugs — fork/upgrade instead |
| APM/observability agents | Thread-safe global state — it's a global mutation |
| Emergency prod hotfixes | When subclassing or composition is cleaner |

### Key Tradeoffs

**Invisible behavior** — When something breaks, stack traces point to the patched function, not the patch. New engineers have no idea a patch exists.

**Version fragility** — Your patch targets internal class methods. Library upgrades silently break patches. No compile-time warning.

**Test pollution** — Unscoped patches persist across tests. Causes flaky tests that fail only in certain run orders.

**Thread safety** — Patching a class attribute is a global write. Concurrent patch + read = race condition.

---

## 8. Comparison

### Monkey Patching vs Dependency Injection vs Subclassing

| Aspect | Monkey Patching | Dependency Injection | Subclassing |
|---|---|---|---|
| Code change needed | No | Yes | Yes |
| Scope | Global process-wide | Per instance | Per class hierarchy |
| Auto-restore | Manual / `mock.patch` | Just pass different object | Not needed |
| Debuggability | Hard — invisible | Excellent — explicit | Good |
| Third-party libs | Works — only option often | Requires lib support | Often impossible |
| Best for | Testing, APM, gevent | New code design | Extending behavior cleanly |
| Production safety | Risky if misused | Safe | Safe |

**Rule of thumb:** For new code you control → DI. For tests → `mock.patch`. For third-party or concurrency → monkey patch.

---

## 9. Production Debugging / Failure Cases

### Failure 1: Mock Not Working (Most Common)

**Symptom:** Real function is being called in tests. Network calls happening.

**Cause:** Wrong patch target — patched where defined, not where used.

**Debug:**
```python
@patch('requests.post')           # wrong if payment.py does: from requests import post
def test_payment(mock_post):
    process_payment(100)
    mock_post.assert_called_once() # FAILS — real requests.post was called
```
**Fix:** `@patch('payment_service.post')`

---

### Failure 2: Test Pollution — Tests Fail in CI, Pass Locally

**Symptom:** Tests pass alone, fail together. Order-dependent failures.

**Cause:** Patch applied without cleanup — mutated module state bleeds into next test.

```python
# WRONG — permanent patch, no restore
def test_something():
    import payment_service
    payment_service.client.charge = MagicMock()  # never restored!
```

**Fix:** Always use `@patch` or `with patch(...)` — they restore automatically.

---

### Failure 3: gevent Blocking Despite Patching

**Symptom:** Concurrency not achieved. Workers behave serially under load.

**Cause:** `patch_all()` called after a library already imported socket.

**Debug:**
```python
import gevent.monkey
print(gevent.monkey.is_module_patched('socket'))  # False = problem
```

**Fix:** `patch_all()` must be the absolute first import in `wsgi.py`.

---

### Failure 4: APM Tracing Breaks After Library Upgrade

**Symptom:** After upgrading `psycopg2`, DB queries no longer appear in Datadog traces.

**Cause:** APM agent patched internal class methods that changed in the new version.

**Fix:** Pin library versions when APM agents are known to depend on internals. Check APM agent release notes before upgrading DB drivers.

---

## 10. Key Code Examples

### Standard Test Mock Pattern

```python
from unittest.mock import patch, MagicMock
from sqlalchemy.orm import Session

@patch('order_service.db_session_factory')
@patch('order_service.notification_client.send')
def test_place_order(mock_notify, mock_session_factory):
    mock_session = MagicMock(spec=Session)
    mock_session_factory.return_value.__enter__.return_value = mock_session

    order_service.place_order(user_id="U001", items=[{"id": "I1", "qty": 2}])

    mock_session.add.assert_called_once()
    mock_session.commit.assert_called_once()
    mock_notify.assert_called_once()
```

### Patching `os.environ` (Common in Auth/Config Tests)

```python
@patch.dict(os.environ, {
    'JWT_SECRET': 'test-secret-key',
    'TOKEN_EXPIRY_HOURS': '24',
    'ENV': 'test'
})
def test_token_generation():
    token = auth_service.generate_token(user_id="U001")
    payload = auth_service.decode_token(token)
    assert payload["user_id"] == "U001"
```

### Simulating Failures with `side_effect`

```python
@patch('app.cache.redis_client.get')
def test_fallback_on_cache_failure(mock_redis_get):
    # Simulate Redis being down
    mock_redis_get.side_effect = redis.ConnectionError("Redis unreachable")

    # Service should fall back to DB, not crash
    result = user_service.get_profile("U001")
    assert result is not None  # DB fallback worked
```

### Emergency Hotfix Pattern

```python
# hotfix.py — loaded at app startup before anything else
import buggy_lib
import logging

logger = logging.getLogger(__name__)

_original = buggy_lib.QuerySet.find

def patched_find(self, *args, **kwargs):
    """Hotfix: adds missing LIMIT. Remove after upgrading to buggy_lib>=3.2"""
    kwargs.setdefault('limit', 1000)
    return _original(self, *args, **kwargs)

buggy_lib.QuerySet.find = patched_find
logger.critical("Hotfix applied: buggy_lib.QuerySet.find | Ticket: INC-4521")
```

---

## 11. Revision Notes — Interview Priority

### Must Know

| Concept | One-liner |
|---|---|
| What it is | Reassigning names in module/class `__dict__` at runtime |
| Why it works | Python namespaces are mutable dicts |
| `mock.patch` | Scoped replacement with auto-restore — use for ALL test mocking |
| Patch where USED | `from x import f` → patch `your_module.f`, not `x.f` |
| `spec=` in MagicMock | Prevents phantom attribute bugs in tests |
| `patch.dict` | For `os.environ`, config dicts |
| gevent `patch_all()` | Must be first import; replaces stdlib I/O for concurrency |
| APM patching | Datadog/New Relic auto-instrument without code change |

### Most Likely Interview Question

> *"You're writing tests for a service that calls an external payment API. How do you test it without making real API calls?"*

**Answer flow:** `mock.patch` → correct patch target (where used) → `MagicMock(spec=...)` → assert calls + return values → test failure scenarios with `side_effect`.

That's the whole topic from an interview standpoint.

---

> **Final Tip:** For a backend role interview, monkey patching = `mock.patch` + knowing the "patch where used" rule + one sentence on gevent. Don't overcomplicate it. The interviewer wants to know you can write testable backend code — that's the real signal.
